# Assignment 5 - Reranker and Consolidator

## FP3 Not in Context - Consolidation strategy Limitations
Documents with the answer were retrieved from the database but did not make it into the context for generating an answer. This occurs when many documents are returned from the database and a consolidation process takes place to retrieve the answer.


## Goal
This notebook will address the reranking and consolidation steps of the RAG system

### Reranking
Cross Encoder, MMR (Maximal Marginal Relevance), Reciprocal Rank Fusion


### Consolidation
Possible include an LLM call?


In [1]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia

!pip install rouge-score

In [19]:
import os
import numpy as np
import time
import locale
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')


# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

COHERE_API_KEY = userdata.get('COHERE_API_KEY')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

from rouge_score import rouge_scorer


import os
os.environ["USER_AGENT"] = "RAG_Assignment/v0.1 (davidschaaf@berkeley.edu)"

# 2. Restore your data at the start of a new session
!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_40.tar.gz" -C /content/qdrant_storage/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [3]:
%%capture
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import linear_kernel # dot product
EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
base_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL)

In [29]:
vector_store = QdrantVectorStore(
    client=QdrantClient(path="/content/qdrant_storage"),
    embedding=base_embeddings,
    collection_name="rag_tech_db",
    distance=Distance.DOT
)


In [ ]:
# %%capture
# quantization_config = BitsAndBytesConfig(
#    load_in_4bit=True,
#    bnb_4bit_quant_type="nf4",
#    bnb_4bit_use_double_quant=True,
#    bnb_4bit_compute_dtype=torch.bfloat16
# )

# llm_mistral_model = AutoModelForCausalLM.from_pretrained(
#     "mistralai/Mistral-7B-Instruct-v0.3",
#     dtype=torch.float32,
#     device_map='auto',
#     quantization_config=quantization_config
# )

# llm_mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

# mistral_pipe = pipeline(
#     "text-generation",
#     model=llm_mistral_model,
#     tokenizer=llm_mistral_tokenizer,
#     max_new_tokens=1000,
#     temperature=0.6,
#     top_p=0.95,
#     do_sample=True,
#     repetition_penalty=1.2
# )

# mistral_pipe.model.config.pad_token_id = mistral_pipe.model.config.eos_token_id

# mistral_llm_lc = HuggingFacePipeline(pipeline=mistral_pipe)

In [34]:
%%capture
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", activation_fn=torch.nn.Sigmoid()))

In [ ]:
marketing_persona = {
    "name": "marketing",
    "description": """This user is a marketer who will ask questions about generative AI in order to better understand the products and the field as a whole.
                      They prefer high level answers that explain concept over technical detail.
                      You will help them find accurate, approved messaging about generative AI features, competitive positioning, and technical capabilities to accelerate content production.""",
}

research_persona = {
    "name": "research",
    "description": """This user is an engineer, who requires detailed technical information when they ask questions.
                      You will help them by writing questions about generative AI concepts, internal system architecture, and implementation details."""
}

llm_template = """Write a question that a {name} professional would ask based on the following context.
{description}

Context:
{context}

Respond only with the question nothing else.
Question:"""
llm_prompt_template = PromptTemplate(template=llm_template, input_variables=["object"])

cohere_chat_model = ChatCohere(cohere_api_key=COHERE_API_KEY)

cohere_chain = llm_prompt_template | cohere_chat_model | StrOutputParser()


In [37]:
import json
import json
import sys
from pathlib import Path
import numpy as np

path = "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_output.json"
records = json.loads(Path(path).read_text())
config = Path(path).stem

# {
#   "persona": "research",
#   "question_score": 28.551249964291134,
#   "question_id": "2507.09477",
#   "recall": [
#     1,
#     0,
#     0,
#     0,
#     0
#   ],
#   "question": "How does the Retrieval-Augmented Decision Transformer architecture enhance the capabilities of LLM agents in research assistance tasks?",
#   "id": "2507.09477",
#   "context_id": "0b6d761a9f0a44e88055cde833633dbc"
# }


In [43]:
import json
import sys
from pathlib import Path
import numpy as np

import random
random.seed(42)

PERSONAS = ["research", "marketing"]

def eval_context_persona_reranked(record):
    question=record['question']
    document_id = record['id']
    vector_store_result = [doc for doc, score, in vector_store.similarity_search_with_score(question, k=20)]
    scores = cross_encoder.predict([(question, doc.page_content) for doc in vector_store_result])
    reranked_docs = [d for score, d in sorted(zip(scores, vector_store_result), key=lambda x: x[0], reverse=True)][:5]

    recall = [1 if x.metadata['id'] == document_id else 0 for x in reranked_docs]
    score = np.mean(scores)
    result = record.copy()
    result['recall'] = recall
    result['question_score'] = score
    return result

points, _ = vector_store.client.scroll(
    collection_name=vector_store.collection_name,
    limit=1000, with_payload=True)

output = []
research_scores = []
marketing_scores = []
research_recall_1 = []
research_recall_3 = []
research_recall_5 = []
marketing_recall_1 = []
marketing_recall_3 = []
marketing_recall_5 = []

for record in records:

    result = eval_context_persona_reranked(record)
    # Convert np.float32 to float before appending to output
    result['question_score'] = float(result['question_score'])
    output.append(result)
    if record['persona'] == 'research':
        research_scores.append(result['question_score'])
        research_recall_1.append(any(result['recall'][:1]))
        research_recall_3.append(any(result['recall'][:3]))
        research_recall_5.append(any(result['recall'][:5]))
    else:
        marketing_scores.append(result['question_score'])
        marketing_recall_1.append(any(result['recall'][:1]))
        marketing_recall_3.append(any(result['recall'][:3]))
        marketing_recall_5.append(any(result['recall'][:5]))

path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_crossencoder_output.json"
json.dump(output, open(path, "w"), indent=2)

print(f"Average Research Score = {np.mean(research_scores):.3f}") # Does not compare to Similarity Search Score
print(f"Average Marketing Score = {np.mean(marketing_scores):.3f}") # Does not compare to Similarity Search Score
print(f"Average Research Recall@1 = {np.mean(research_recall_1):.3f}")
print(f"Average Research Recall@3 = {np.mean(research_recall_3):.3f}")
print(f"Average Research Recall@5 = {np.mean(research_recall_5):.3f}")
print(f"Average Marketing Recall@1 = {np.mean(marketing_recall_1):.3f}")
print(f"Average Marketing Recall@3 = {np.mean(marketing_recall_3):.3f}")
print(f"Average Marketing Recall@5 = {np.mean(marketing_recall_5):.3f}")

Average Research Score = -4.118
Average Marketing Score = -3.925
Average Research Recall@1 = 0.830
Average Research Recall@3 = 0.890
Average Research Recall@5 = 0.920
Average Marketing Recall@1 = 0.590
Average Marketing Recall@3 = 0.640
Average Marketing Recall@5 = 0.720


In [48]:

PERSONAS = ["research", "marketing"]


def summarize(records, persona):
    rows = [r for r in records if r.get("persona") == persona]
    if not rows:
        return None

    scores = [r["question_score"] for r in rows]

    # recall@k: did the gold doc appear anywhere in the top k?
    def recall_at(k):
        return np.mean([any(r["recall"][:k]) for r in rows])

    # density: what fraction of the top-5 came from the gold doc?
    density = np.mean([sum(r["recall"]) / len(r["recall"]) for r in rows])

    return {
        "n": len(rows),
        "score": np.mean(scores),
        "r1": recall_at(1),
        "r3": recall_at(3),
        "r5": recall_at(5),
        "density": density,
    }



paths = [
    "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_crossencoder_output.json",
    "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_output.json",
]


def short_label(path):
    """Trim boilerplate off the filename so configs are distinguishable."""
    stem = Path(path).stem
    for junk in ("vector_store_eval_", "_output", "chunk_", "olap_"):
        stem = stem.replace(junk, "")
    return stem  # e.g. "250_40_crossencoder"


header = (
    f"{'config':<24}{'persona':<11}{'n':>5}{'score':>9}"
    f"{'R@1':>7}{'R@3':>7}{'R@5':>7}{'density':>9}"
)
print(header)
print("-" * len(header))

for path in paths:
    records = json.loads(Path(path).read_text())
    config = short_label(path)

    for persona in PERSONAS:
        m = summarize(records, persona)
        if m is None:
            continue
        print(
            f"{config:<24.24}{persona:<11}{m['n']:>5}{m['score']:>9.1f}"
            f"{m['r1']:>7.2f}{m['r3']:>7.2f}{m['r5']:>7.2f}{m['density']:>9.2f}"
        )




config                  persona        n    score    R@1    R@3    R@5  density
-------------------------------------------------------------------------------
250_40_crossencoder     research     100     -4.1   0.83   0.89   0.92     0.55
250_40_crossencoder     marketing    100     -3.9   0.59   0.64   0.72     0.37
250_40                  research     100     30.0   0.76   0.86   0.89     0.49
250_40                  marketing    100     28.6   0.56   0.67   0.69     0.35


## Test of Cross Encoder

I reran the same test as the Vector Store Hyperparameters.

The Cross Encoder improved results when given 20 chunks to rerank.

| Config | Persona | n | Score | R@1 | R@3 | R@5 | Density |
|---|---|---:|---:|---:|---:|---:|---:|
| 250/40 + CrossEncoder | Research | 100 | -4.1 | 0.83 | 0.89 | 0.92 | 0.55 |
| 250/40 + CrossEncoder | Marketing | 100 | -3.9 | 0.59 | 0.64 | 0.72 | 0.37 |
| 250/40 baseline | Research | 100 | 30.0 | 0.76 | 0.86 | 0.89 | 0.49 |
| 250/40 baseline | Marketing | 100 | 28.6 | 0.56 | 0.67 | 0.69 | 0.35 |

In [52]:
path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_crossencoder_output.json"

cross_records = json.loads(Path(path).read_text())

In [56]:
threshold = 0.5 # 0-1 for sigmoid activation fn

for record in cross_records:
    question=record['question']
    document_id = record['id']
    vector_store_result = [doc for doc, score, in vector_store.similarity_search_with_score(question, k=50)]
    scores = cross_encoder.predict([(question, doc.page_content) for doc in vector_store_result])
    reranked_docs = [(score, d) for score, d in sorted(zip(scores, vector_store_result), key=lambda x: x[0], reverse=True)]

    unfiltered = reranked_docs[:5]
    recall_unfiltered = [1 if x[1].metadata['id'] == document_id else 0 for x in unfiltered][:5]

    filtered_docs = list(filter(lambda reranked: reranked[0] >= threshold, reranked_docs))
    recall_filtered = [1 if x[1].metadata['id'] == document_id else 0 for x in filtered_docs][:5]

    record['recall_unfiltered'] = recall_unfiltered
    record['recall_filtered'] = recall_filtered





In [57]:
import numpy as np
from collections import defaultdict

def bundle_metrics(recall_list):
    """precision (density), recall (any hit), size for one bundle."""
    n = len(recall_list)
    if n == 0:
        return 0.0, 0.0, 0            # empty bundle: no precision, no hit
    return sum(recall_list) / n, float(any(recall_list)), n

def summarize(records, key, label):
    by_persona = defaultdict(list)
    for r in records:
        by_persona[r['persona']].append(bundle_metrics(r[key]))

    for persona, vals in by_persona.items():
        prec = np.mean([v[0] for v in vals])
        rec  = np.mean([v[1] for v in vals])
        size = np.mean([v[2] for v in vals])
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        empty = sum(1 for v in vals if v[2] == 0)
        print(f"{label:<14}{persona:<11}{len(vals):>5}"
              f"{prec:>11.3f}{rec:>9.3f}{f1:>7.3f}{size:>7.2f}{empty:>8}")

header = (f"{'condition':<14}{'persona':<11}{'n':>5}"
          f"{'precision':>11}{'recall':>9}{'F1':>7}{'size':>7}{'empty':>8}")
print(header)
print("-" * len(header))

summarize(cross_records, 'recall_unfiltered', 'no filter')
summarize(cross_records, 'recall_filtered',  f'tau={threshold}')

condition     persona        n  precision   recall     F1   size   empty
------------------------------------------------------------------------
no filter     research     100      0.524    0.940  0.673   5.00       0
no filter     marketing    100      0.386    0.770  0.514   5.00       0
tau=0.5       research     100      0.673    0.870  0.759   2.71       9
tau=0.5       marketing    100      0.535    0.680  0.599   2.55      10


## Filter Results after Cross Encoder

